# 07 - Masked v2 Deployment Training (Fase 2)

Notebook ini melatih ulang model deployment (`rain_7d`, `wind_30d`) dengan **masked sliding
window**: hari/nilai hilang ditandai `mask=0` dan **tidak** dihitung di reconstruction error.
Gap pendek diimputasi hanya untuk input model.

Beda dengan v1: v1 menolak scoring jika window tidak berurutan; v2 tetap menilai selama
rasio data observed mencukupi (`MIN_OBSERVED_RATIO`).

Prasyarat: TensorFlow tersedia dan tabel fitur notebook 01 ada di
`../data/processed/yogyakarta_weather_features.csv`.


In [ ]:
# Workaround image Jupyter: libstdc++ sistem kadang lebih tua daripada yang
# dibutuhkan TensorFlow/protobuf (GLIBCXX_3.4.29). Muat libstdc++ conda dulu.
import ctypes
import os

for candidate in (
    os.path.join(os.environ.get("CONDA_PREFIX", "") or "", "lib", "libstdc++.so.6"),
    "/opt/conda/lib/libstdc++.so.6",
    "/usr/local/lib/libstdc++.so.6",
):
    if candidate and os.path.exists(candidate):
        try:
            ctypes.CDLL(candidate, mode=ctypes.RTLD_GLOBAL)
            print("preloaded libstdc++:", candidate)
        except OSError as error:
            print("skip", candidate, "->", error)
        break


## 1. Setup


In [ ]:
from pathlib import Path
import json
import random
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "yogyakarta_weather_features.csv"
OUT_ROOT = PROJECT_ROOT / "artifacts" / "deployment_v2"

SEED = 42
EPOCHS = 80
PATIENCE = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
MIN_OBSERVED_RATIO = 0.7
IMPUTE_MAX_GAP_DAYS = 3

random.seed(SEED)
np.random.seed(SEED)

print("project root:", PROJECT_ROOT)
print("processed   :", PROCESSED_PATH)
print("out dir     :", OUT_ROOT)


In [ ]:
import tensorflow as tf

tf.random.set_seed(SEED)
print("tensorflow:", tf.__version__)


## 2. Load data & konfigurasi model


In [ ]:
from app.masking import (
    build_feature_masks,
    impute_window,
    masked_weighted_error,
    observed_ratio,
    to_calendar_window,
)

data = pd.read_csv(PROCESSED_PATH, parse_dates=["date"], dtype={"station_id": "string"})
data = data.sort_values(["station_id", "date"]).reset_index(drop=True)
print("rows:", len(data), "| stations:", sorted(data["station_id"].astype(str).unique()))
display(data.head())

DEPLOYMENT_MODELS = [
    {
        "experiment_id": "rain_7d",
        "hazard": "curah_hujan_tinggi",
        "sequence_length": 7,
        "feature_group": "rain_only",
        "features": ["RR", "rain_3d", "rain_7d", "rain_change_1d", "missing_RR"],
        "weights": {"RR": 3.0, "rain_3d": 3.0, "rain_7d": 3.0, "rain_change_1d": 2.0, "missing_RR": 0.5},
    },
    {
        "experiment_id": "wind_30d",
        "hazard": "angin_kencang",
        "sequence_length": 30,
        "feature_group": "wind_only",
        "features": ["ff_x", "ff_avg", "wind_change_1d", "ddd_x_sin", "ddd_x_cos", "missing_ff_x", "missing_ff_avg"],
        "weights": {"ff_x": 3.0, "ff_avg": 3.0, "wind_change_1d": 3.0, "ddd_x_sin": 1.0, "ddd_x_cos": 1.0, "missing_ff_x": 0.5, "missing_ff_avg": 0.5},
    },
]


## 3. Helper: window kalender + mask


In [ ]:
def calendar_frames(data, features, impute_max_gap_days):
    frames = {}
    for station_id, group in data.groupby("station_id", sort=False):
        group = group.sort_values("date").reset_index(drop=True)
        span = (group["date"].max() - group["date"].min()).days + 1
        window = to_calendar_window(group, span)
        masks = build_feature_masks(window, features)
        imputed = impute_window(window, features, impute_max_gap_days)
        frames[str(station_id)] = (window["date"], imputed[features].astype(float), masks)
    return frames


def chronological_split_dates(data, train_ratio=0.70, val_ratio=0.85):
    dates = np.array(sorted(pd.to_datetime(data["date"]).dt.normalize().unique()))
    train_end = int(len(dates) * train_ratio)
    val_end = int(len(dates) * val_ratio)
    to_ts = lambda values: {pd.Timestamp(value) for value in values}
    return to_ts(dates[:train_end]), to_ts(dates[train_end:val_end]), to_ts(dates[val_end:])


def build_windows(frames, features, sequence_length, scaler, date_sets):
    train_dates, val_dates, test_dates = date_sets
    x_parts, m_parts, meta = [], [], []
    for station_id, (dates, values, masks) in frames.items():
        scaled = pd.DataFrame(scaler.transform(values), columns=features)
        for end in range(sequence_length - 1, len(dates)):
            end_date = pd.Timestamp(dates.iloc[end])
            if end_date in train_dates:
                split = "train"
            elif end_date in val_dates:
                split = "validation"
            elif end_date in test_dates:
                split = "test"
            else:
                continue
            start = end - sequence_length + 1
            x_parts.append(scaled.iloc[start:end + 1].to_numpy(dtype=np.float32))
            m_parts.append(masks.iloc[start:end + 1].to_numpy(dtype=np.float32))
            meta.append({"split": split, "date": end_date, "station_id": station_id})
    return np.stack(x_parts), np.stack(m_parts), pd.DataFrame(meta)


def normalised_weights(features, source):
    raw = np.array([source.get(name, 1.0) for name in features], dtype=np.float32)
    return raw / raw.mean()


## 4. Model & masked training


In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam


def build_compact_lstm_autoencoder(sequence_length, n_features, learning_rate=0.001):
    inputs = Input(shape=(sequence_length, n_features))
    encoded = LSTM(32, activation="tanh", return_sequences=True)(inputs)
    encoded = Dropout(0.15)(encoded)
    encoded = LSTM(16, activation="tanh", return_sequences=False)(encoded)
    decoded = RepeatVector(sequence_length)(encoded)
    decoded = LSTM(16, activation="tanh", return_sequences=True)(decoded)
    decoded = Dropout(0.15)(decoded)
    decoded = LSTM(32, activation="tanh", return_sequences=True)(decoded)
    outputs = TimeDistributed(Dense(n_features))(decoded)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse")
    return model


def masked_loss_values(x, recon, mask, weights):
    weights_t = tf.reshape(tf.constant(weights, dtype=tf.float32), (1, 1, -1))
    numerator = tf.reduce_sum(tf.square(x - recon) * weights_t * mask, axis=[1, 2])
    denominator = tf.reduce_sum(weights_t * mask, axis=[1, 2]) + 1e-8
    return numerator / denominator


def evaluate_masked_loss(model, x, m, weights):
    recon = model(x, training=False)
    return float(tf.reduce_mean(masked_loss_values(x, recon, m, weights)))


def train_masked(model, x, m, weights, x_val, m_val, epochs, patience, batch_size, learning_rate):
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    x = tf.constant(x)
    m = tf.constant(m)
    x_val = tf.constant(x_val)
    m_val = tf.constant(m_val)
    best = float("inf")
    best_weights = None
    stale = 0
    history = []
    for epoch in range(1, epochs + 1):
        order = np.random.permutation(int(x.shape[0]))
        losses = []
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            xb, mb = tf.gather(x, idx), tf.gather(m, idx)
            with tf.GradientTape() as tape:
                recon = model(xb, training=True)
                loss = tf.reduce_mean(masked_loss_values(xb, recon, mb, weights))
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            losses.append(float(loss))
        train_loss = float(np.mean(losses))
        val_loss = evaluate_masked_loss(model, x_val, m_val, weights)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"epoch {epoch:3d} | train {train_loss:.6f} | val {val_loss:.6f}")
        if val_loss < best - 1e-6:
            best = val_loss
            best_weights = [v.numpy().copy() for v in model.trainable_variables]
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                print(f"early stop at epoch {epoch} (patience {patience})")
                break
    if best_weights is not None:
        for variable, value in zip(model.trainable_variables, best_weights):
            variable.assign(value)
    return history


## 5. Train & export artifact v2


In [ ]:
def thresholds_from(values):
    clean = values[~np.isnan(values)]
    return {
        "p95": float(np.percentile(clean, 95)),
        "p99": float(np.percentile(clean, 99)),
        "p995": float(np.percentile(clean, 99.5)),
    }


def evaluate_errors(model, x, m, weights):
    recon = model.predict(x, verbose=0)
    totals, per_feature = [], []
    for i in range(x.shape[0]):
        total, feature_errors = masked_weighted_error(x[i], recon[i], weights, m[i])
        totals.append(np.nan if total is None else total)
        per_feature.append([np.nan if feature_errors[j] is None else feature_errors[j] for j in range(x.shape[2])])
    return np.array(totals), np.array(per_feature)


summary_rows = []
for config in DEPLOYMENT_MODELS:
    features = config["features"]
    weights = normalised_weights(features, config["weights"])
    print("=" * 70)
    print(f"Training {config['experiment_id']} | window {config['sequence_length']} | {len(features)} features")

    frames = calendar_frames(data, features, IMPUTE_MAX_GAP_DAYS)
    date_sets = chronological_split_dates(data)

    train_rows = []
    for _station_id, (dates, values, _masks) in frames.items():
        in_train = pd.to_datetime(dates).isin(date_sets[0]).to_numpy()
        if in_train.any():
            train_rows.append(values.to_numpy()[in_train])
    scaler = RobustScaler().fit(pd.DataFrame(np.vstack(train_rows), columns=features))

    x, m, meta = build_windows(frames, features, config["sequence_length"], scaler, date_sets)
    train = (meta["split"] == "train").to_numpy()
    validation = (meta["split"] == "validation").to_numpy()
    test = (meta["split"] == "test").to_numpy()
    print(f"windows: train {int(train.sum())} | validation {int(validation.sum())} | test {int(test.sum())}")

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = build_compact_lstm_autoencoder(config["sequence_length"], len(features), LEARNING_RATE)
    history = train_masked(
        model, x[train], m[train], weights, x[validation], m[validation],
        EPOCHS, PATIENCE, BATCH_SIZE, LEARNING_RATE,
    )

    total_val, per_feature_val = evaluate_errors(model, x[validation], m[validation], weights)
    total_thresholds = thresholds_from(total_val)
    per_feature_thresholds = {
        name: thresholds_from(per_feature_val[:, index]) for index, name in enumerate(features)
    }

    out_dir = OUT_ROOT / config["experiment_id"]
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save(out_dir / "model.keras", include_optimizer=False)
    joblib.dump(scaler, out_dir / "scaler.pkl")
    (out_dir / "threshold.json").write_text(
        json.dumps({"total": total_thresholds, "per_feature": per_feature_thresholds}, indent=2),
        encoding="utf-8",
    )
    (out_dir / "feature_config.json").write_text(
        json.dumps(
            {
                "experiment_id": config["experiment_id"],
                "hazard": config["hazard"],
                "sequence_length": config["sequence_length"],
                "feature_group": config["feature_group"],
                "model_features": features,
                "feature_weights": {name: float(w) for name, w in zip(features, weights)},
                "mask_policy": "observed_mask_v1",
                "min_observed_ratio": MIN_OBSERVED_RATIO,
                "impute_max_gap_days": IMPUTE_MAX_GAP_DAYS,
                "version": 2,
                "source_notebook": "notebooks/07_masked_v2_training.ipynb",
                "notes": "Masked sliding-window deployment artifact.",
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    (out_dir / "training_report.json").write_text(
        json.dumps(
            {
                "experiment_id": config["experiment_id"],
                "train_windows": int(train.sum()),
                "validation_windows": int(validation.sum()),
                "test_windows": int(test.sum()),
                "seed": SEED,
                "epochs_ran": len(history),
                "train_masked_loss_history": history,
                "thresholds": {"total": total_thresholds, "per_feature": per_feature_thresholds},
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"saved -> {out_dir}")
    summary_rows.append(
        {
            "experiment_id": config["experiment_id"],
            "hazard": config["hazard"],
            "sequence_length": config["sequence_length"],
            "feature_group": config["feature_group"],
            "n_features": len(features),
            "train_windows": int(train.sum()),
            "validation_windows": int(validation.sum()),
            "test_windows": int(test.sum()),
            "total_p95": total_thresholds["p95"],
            "total_p99": total_thresholds["p99"],
            "total_p995": total_thresholds["p995"],
            "artifact_dir": str(out_dir),
        }
    )

OUT_ROOT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(summary_rows).to_csv(OUT_ROOT / "deployment_model_export_summary.csv", index=False)
print("Done:", OUT_ROOT / "deployment_model_export_summary.csv")


## 6. Ringkasan & threshold


In [ ]:
from IPython.display import display

display(pd.read_csv(OUT_ROOT / "deployment_model_export_summary.csv"))

for config in DEPLOYMENT_MODELS:
    payload = json.loads((OUT_ROOT / config["experiment_id"] / "threshold.json").read_text(encoding="utf-8"))
    print("=" * 70)
    print(config["experiment_id"], "| total:", payload["total"])
    display(pd.DataFrame(payload["per_feature"]).T)


## 7. Perbandingan threshold v1 vs v2


In [ ]:
rows = []
for config in DEPLOYMENT_MODELS:
    model_id = config["experiment_id"]
    v2 = json.loads((OUT_ROOT / model_id / "threshold.json").read_text(encoding="utf-8"))["total"]
    v1_path = PROJECT_ROOT / "artifacts" / "deployment" / model_id / "threshold.json"
    v1 = {}
    if v1_path.exists():
        raw = json.loads(v1_path.read_text(encoding="utf-8"))
        v1 = raw.get("total", raw)
    for level in ["p95", "p99", "p995"]:
        rows.append(
            {
                "model": model_id,
                "level": level,
                "v1": v1.get(level),
                "v2": v2.get(level),
                "delta": None if level not in v1 else v2[level] - v1[level],
            }
        )
display(pd.DataFrame(rows))


## 8. Uji cepat: service memuat artifact v2


In [ ]:
from app.model_runner import LSTMAutoencoderRunner

station = sorted(data["station_id"].astype(str).unique())[0]
station_df = data[data["station_id"].astype(str) == station].sort_values("date").reset_index(drop=True)
print("station:", station, "| rows:", len(station_df))

for config in DEPLOYMENT_MODELS:
    runner = LSTMAutoencoderRunner(OUT_ROOT / config["experiment_id"])
    result = runner.score_latest(station_df)
    print("=" * 70)
    print(config["experiment_id"], "->", result.status, "| score:", result.score, "| missing_ratio:", result.missing_ratio)
    print("reason:", result.reason)
    if result.feature_errors:
        print("feature errors:", result.feature_errors)


## 9. Promosi ke produksi (setelah divalidasi)

```bash
cp -r artifacts/deployment artifacts/deployment_v1_backup
cp -r artifacts/deployment_v2/* artifacts/deployment/
git add artifacts && git commit -m "chore: promote masked v2 deployment artifacts" && git push
```

Lalu di VPS: `git pull` -> `docker build` -> jalankan ulang container (dry-run dulu).
